# Business Questions

Every answer is built from the Gold layer only, as required by item 4 of the
case. The source files are never read here.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
SOURCE_CATALOG = 'beverage_sales'
SOURCE_SCHEMA = 'gold'

In [0]:
df_agg_sales_region_trade_group = spark.table(f'{SOURCE_CATALOG}.{SOURCE_SCHEMA}.agg_sales_region_trade_group').alias('artg')
df_agg_sales_brand_month = spark.table(f'{SOURCE_CATALOG}.{SOURCE_SCHEMA}.agg_sales_brand_month').alias('absm')
df_agg_sales_region_brand_month = spark.table(f'{SOURCE_CATALOG}.{SOURCE_SCHEMA}.agg_sales_region_brand_month').alias('arbm')
df_dim_region = spark.table(f'{SOURCE_CATALOG}.{SOURCE_SCHEMA}.dim_region').alias('dr')
df_dim_brand = spark.table(f'{SOURCE_CATALOG}.{SOURCE_SCHEMA}.dim_brand').alias('db')

## Question 4.1 - What are the Top 3 Trade Groups (TRADE_GROUP_DESC) for each Region (Btlr_Org_LVL_C_Desc) in sales ($ Volume)?

In [0]:
window_region = Window.partitionBy('region').orderBy(
    F.desc('dollar_volume')
)

df_answer_4_1 = (
    df_agg_sales_region_trade_group
    .withColumn('rank', F.dense_rank().over(window_region))
    .filter(F.col('rank') <= 3)
    .select(
        'region',
        'rank',
        'trade_group',
        'dollar_volume',
        'record_count'
    )
    .orderBy('region', 'rank')
)

display(df_answer_4_1)

## Question 4.2 - How much sales ($ Volume) each brand (BRAND_NM) achieved per month?

In [0]:
df_answer_4_2 = (
    df_agg_sales_brand_month
    .select(
        'year',
        'month',
        'month_name',
        'year_month',
        'brand_name',
        'dollar_volume',
        'record_count'
    )
    .orderBy('year', 'month', F.desc('dollar_volume'))
)

display(df_answer_4_2)

## Question 4.3 - Which are the lowest brand (BRAND_NM) in sales ($ Volume) for each region (Btlr_Org_LVL_C_Desc)?

In [0]:
df_region_brand_totals = (
    df_agg_sales_region_brand_month
    .groupBy('region', 'brand_name')
    .agg(F.sum('dollar_volume').alias('dollar_volume'))
)

In [0]:
window_region_brand = Window.partitionBy('region').orderBy(
    F.asc('dollar_volume')
)

df_answer_4_3_a = (
    df_region_brand_totals
    .withColumn('rank', F.dense_rank().over(window_region_brand))
    .filter(F.col('rank') == 1)
    .select(
        'region',
        'brand_name',
        'dollar_volume'
    )
    .orderBy('region')
)

display(df_answer_4_3_a)